# Battleship AI 🚢
Random shooting needs ~95 shots to sink a fleet on a 10×10 board. A smart player reasons: *where could the remaining ships still be?* Count every legal placement of every remaining ship, shoot where the most placements overlap (a **probability density**), and weight placements that explain existing hits much higher. That's Bayesian reasoning by counting.

Background: [[Bayesian Inference]], [[Probability for ML]] · overview: [[Fun Projects]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
N, FLEET = 10, [5, 4, 3, 3, 2]
UNKNOWN, MISS, HIT, SUNK = 0, 1, 2, 3
def random_fleet(rng):
    board = np.zeros((N, N), int); ships = []
    for k, L in enumerate(FLEET):
        while True:
            r, c, horiz = rng.integers(N), rng.integers(N), rng.random() < 0.5
            cells = [(r, c + i) if horiz else (r + i, c) for i in range(L)]
            if all(0 <= i < N and 0 <= j < N and board[i, j] == 0 for i, j in cells):
                for i, j in cells: board[i, j] = k + 1
                ships.append(cells); break
    return board, ships
def play(strategy, rng):
    board, ships = random_fleet(rng); shots = np.zeros((N, N), int); remaining = list(FLEET); n = 0
    while remaining:
        r, c = strategy(shots, remaining, rng); n += 1
        if board[r, c]:
            shots[r, c] = HIT; ship = ships[board[r, c] - 1]
            if all(shots[i, j] == HIT for i, j in ship):
                for i, j in ship: shots[i, j] = SUNK
                remaining.remove(len(ship))
        else: shots[r, c] = MISS
    return n

## 1. Where can a ship of length L still be?
Yield every horizontal and vertical placement (a list of `L` cells) that stays on the board and covers no MISS and no SUNK cell.

In [ ]:
def placements(shots, length):
    # TODO 1: generator over legal placements
    raise NotImplementedError  # TODO 1

## 2. The density map
For every remaining ship length and every legal placement, add a weight to each of its cells: 1, **plus 20 per HIT cell the placement covers** (hits are strong evidence). Then zero out cells already shot. Return an `N×N` array.

In [ ]:
def density(shots, remaining):
    # TODO 2: weighted placement counts per cell
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_bs():
    s = np.zeros((N, N), int); s[0, 1] = MISS
    p = list(placements(s, 3)); d = density(s, [2])
    s2 = np.zeros((N, N), int); s2[5, 5] = HIT; d2 = density(s2, [2])
    return all((0, 1) not in cells for cells in p) and len(p) == 2 * 10 * 8 - 3 and d[0, 1] == 0 and d2[5, 5] == 0 and d2[5, 6] > d2[0, 0] * 10
check("density", _t_bs, "Placements avoid misses (157 legal 3-ships here); cells next to a hit must be much denser than a corner.")
def random_strategy(shots, remaining, rng):
    free = np.argwhere(shots == UNKNOWN); return tuple(free[rng.integers(len(free))])
def hunt_strategy(shots, remaining, rng):
    d = density(shots, remaining); best = np.argwhere(d == d.max()); return tuple(best[rng.integers(len(best))])
if ready("density"):
    rng = np.random.default_rng(0)
    rand = [play(random_strategy, rng) for _ in range(50)]; smart = [play(hunt_strategy, rng) for _ in range(50)]
    print(f"average shots to sink the fleet: random {np.mean(rand):.1f}, probability density {np.mean(smart):.1f}")
    plt.hist([rand, smart], bins=range(20, 101, 4), label=["random", "density"]); plt.xlabel("shots"); plt.legend(); plt.show()
    s = np.zeros((N, N), int); s[4, 4] = HIT; s[2, 7] = MISS
    plt.imshow(density(s, FLEET), cmap="magma"); plt.title("density after one hit and one miss"); plt.colorbar(); plt.show()
    check("smart hunter", lambda: np.mean(smart) < 0.7 * np.mean(rand), "The density strategy should need at least 30 % fewer shots than random.")

<details><summary>▶ Solution</summary>

```python
def placements(shots, length):
    for r in range(N):
        for c in range(N):
            for dr, dc in ((0, 1), (1, 0)):
                cells = [(r + k * dr, c + k * dc) for k in range(length)]
                if all(0 <= i < N and 0 <= j < N and shots[i, j] != MISS and shots[i, j] != SUNK for i, j in cells):
                    yield cells
```

```python
def density(shots, remaining):
    dens = np.zeros((N, N))
    hits = list(zip(*np.where(shots == HIT)))
    for length in remaining:
        for cells in placements(shots, length):
            w = 1 + 20 * sum((i, j) in hits for i, j in cells)
            for i, j in cells:
                dens[i, j] += w
    dens[shots != UNKNOWN] = 0
    return dens
```
</details>

## Stretch goals
- Parity trick: the smallest ship has length 2, so in *hunt* mode only shoot black squares of a checkerboard.
- Play against the AI yourself: print the board and type coordinates.
- Exact posterior: sample full fleets consistent with all shots (rejection sampling) and shoot where ships appear most often.